# Embeddings: from words to vectors, and what they are good for

Session 13 ended with the limits of word counts: "Spielzeug", "jouet" and "toy" share no column, and word order is mostly lost. Language models address both by representing text as dense vectors learned from very large amounts of unlabelled text, in many languages at once. The first half of this notebook explains, at a conceptual level, how they do it: subword tokens, embeddings, attention and pre-training, the three families of transformer models, and sentence-embedding models, which turn a whole description of goods into one vector. The second half uses those vectors for three tasks that recur in almost every text project: **semantic search**, **clustering** and **classification features**, each compared with the TF-IDF representation of Session 13 on the same data. The short answer for the case study: embeddings are clearly better for search across languages, useful for exploration, and *worse* than TF-IDF as classification features. The mathematics of neural networks belongs to module 3.4; here the aim is to understand what these models do, what goes in and what comes out.

The code cells build on each other; run them in order. The first cell moves to the repository root, where the case-study paths start. The notebook uses the multilingual model `intfloat/multilingual-e5-small` (about 470 MB, downloaded on the first run and cached afterwards) and the `tiktoken` tokeniser. The cell that encodes 12,000 descriptions takes about a minute on a laptop. No API key is needed.

```mermaid
flowchart LR
    T["Text"] --> K["Tokeniser<br/>(subword tokens)"]
    K --> I["Token ids<br/>[0, 79451, ...]"]
    I --> E["Embedding lookup<br/>(one vector per token)"]
    E --> L["Transformer layers<br/>(attention + feed-forward)"]
    L --> C["Contextual vectors<br/>(one per token)"]
    C --> P1["Pooling: one vector<br/>per text (embeddings)"]
    C --> P2["Next-token scores<br/>(text generation)"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Tokens: how a language model splits text

### Concept

A language model has a fixed vocabulary of **subword tokens**, typically 30,000 to 250,000 entries. Frequent words are single tokens ("plastic"); rare words and compounds are split into pieces ("Kunststoffspielzeugauto" → "Kunststoff" + "spiel" + "zeug" + "auto"). Because single characters are also in the vocabulary, *every* string can be represented: there are no out-of-vocabulary words as in Session 13.

Two families of tokenisers are common:

- **SentencePiece** (multilingual encoders such as XLM-RoBERTa and multilingual-e5): learned on text in about 100 languages; a piece that starts a new word is marked with `▁`.
- **Byte-pair encoding** (BPE; GPT-type models): starts from single bytes and repeatedly merges the most frequent adjacent pair into a new token, until the vocabulary has the desired size (Sennrich, Haddow and Birch, 2016). A leading space is part of the token (" toy").

Worked example of BPE training on a tiny corpus: "low" (3 times), "lower" (once), "newest" (twice).

| Step | Most frequent pair (count) | Merge | Words afterwards |
|---|---|---|---|
| 0 | | | l o w (×3), l o w e r, n e w e s t (×2) |
| 1 | `l o` (4) | `lo` | lo w (×3), lo w e r, n e w e s t (×2) |
| 2 | `lo w` (4) | `low` | low (×3), low e r, n e w e s t (×2) |

After two merges, "low" is a single token and "lower" is `low` + `e` + `r`. Repeated tens of thousands of times on a large corpus, this yields tokens for frequent words and frequent word pieces; German compounds are split into their parts, which the character n-grams of Session 13 approximated.

![The phrase 'plastic toy car with wheels' as 5 words, 6 SentencePiece tokens and 5 BPE tokens; the German 'Kunststoffspielzeugauto mit Rädern' as 3 words, 7 SentencePiece and 9 BPE tokens](../figures/tokenisation.png)

### Why it matters

Tokens are the unit of everything that follows: the model's input limit (the **context window**), the price of an API call and the processing time are all counted in tokens, not words. The descriptions of goods need about two tokens per word. Languages differ: with the GPT-4o tokeniser, one token covers 3.8 characters of French and 3.7 of German but only 2.9 of Polish and 2.7 of Czech (cell below), so the same content costs about a third more in Czech.

### How it works in Python

In [2]:
import pandas as pd
import tiktoken
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")   # 470 MB, first run only
sp_tok = model.tokenizer                                   # SentencePiece tokeniser of the model
gpt_tok = tiktoken.get_encoding("o200k_base")              # BPE tokeniser of GPT-4o-type models

for text in ["Kunststoffspielzeug in Form eines Autos", "plastic toy in the shape of a car"]:
    print(sp_tok.tokenize(text))
    print([gpt_tok.decode([i]) for i in gpt_tok.encode(text)])
# ['▁Kunststoff', 'spiel', 'zeug', '▁in', '▁Form', '▁eines', '▁Auto', 's']
# ['K', 'unst', 'stoff', 'spiel', 'zeug', ' in', ' Form', ' eines', ' Autos']
# ['▁plastic', '▁toy', '▁in', '▁the', '▁shape', '▁of', '▁a', '▁car']
# ['plastic', ' toy', ' in', ' the', ' shape', ' of', ' a', ' car']

# token counts of 2,000 descriptions
decisions = pd.read_parquet("case-study/data/train_sample.parquet")
sample = decisions.sample(2000, random_state=0)
counts = pd.DataFrame({
    "language": sample["language"],
    "characters": sample["description"].str.len(),
    "words": sample["description"].str.split().str.len(),
    "xlmr": sample["description"].map(lambda t: len(sp_tok.tokenize(t))),
    "o200k": sample["description"].map(lambda t: len(gpt_tok.encode(t))),
})
print(counts[["words", "xlmr", "o200k"]].sum().to_dict())
# {'words': 181130, 'xlmr': 345535, 'o200k': 357514}: about two tokens per word
per_lang = counts.groupby("language")[["characters", "o200k"]].sum()
print((per_lang["characters"] / per_lang["o200k"]).loc[["de", "fr", "en", "pl", "cs"]].round(1).to_dict())
# {'de': 3.7, 'fr': 3.8, 'en': 3.2, 'pl': 2.9, 'cs': 2.7}: characters per token
print(model.max_seq_length, round((counts["xlmr"] > model.max_seq_length - 2).mean(), 3))
# 512 0.007: fewer than 1 % of descriptions are longer than the model's input limit

.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18830.18it/s]

['▁Kunststoff', 'spiel', 'zeug', '▁in', '▁Form', '▁eines', '▁Auto', 's']
['K', 'unst', 'stoff', 'spiel', 'zeug', ' in', ' Form', ' eines', ' Autos']
['▁plastic', '▁toy', '▁in', '▁the', '▁shape', '▁of', '▁a', '▁car']
['plastic', ' toy', ' in', ' the', ' shape', ' of', ' a', ' car']


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (626 > 512). Running this sequence through the model will result in indexing errors


{'words': 181130, 'xlmr': 345535, 'o200k': 357514}
{'de': 3.7, 'fr': 3.8, 'en': 3.2, 'pl': 2.9, 'cs': 2.7}
512 0.007


The English descriptions in the data are often written in capitals, which tokenises less efficiently (3.2 characters per token) than ordinary English text.

### In practice

- OpenAI, Anthropic and Google all bill their language-model APIs per token, separately for input and output tokens; tokeniser libraries such as `tiktoken` let customers count tokens before sending a request.
- Petrov et al. (2023, NeurIPS) measured that for some languages the same content needs up to about 15 times as many tokens as in English, with direct effects on price, speed and the amount of text that fits into the context window.

> **Warning.** Every model has its own tokeniser. Token counts from `tiktoken` are exact only for OpenAI models; for other models use their own tokeniser (for example `model.tokenizer` above, or `AutoTokenizer.from_pretrained(...)` from Hugging Face) or the token counts reported in the API response.

> 📖 **Read more:** [extras/01-hf-course-tokenizers.ipynb](extras/01-hf-course-tokenizers.ipynb) (Hugging Face course): loads a tokeniser with `AutoTokenizer` and goes through tokenising, converting tokens to ids, decoding and saving a tokeniser step by step.
>
> 📖 **Read more:** [extras/02-hf-course-bpe-tokenization.ipynb](extras/02-hf-course-bpe-tokenization.ipynb) (Hugging Face course): implements byte-pair encoding from scratch: learning the merge rules on a small corpus and applying them to tokenise new text.

## Embeddings: words and texts as vectors

### Concept

An **embedding** is a vector of real numbers (typically 300–4,000 numbers) that represents a token, a word or a whole text, learned so that similar items get nearby vectors. A bag-of-words vector of Session 13 has one dimension per vocabulary word, mostly zeros; an embedding is **dense** and **low-dimensional**, and its individual dimensions have no readable meaning.

Similarity between embeddings is measured with **cosine similarity**: cos(a, b) = a·b / (‖a‖ ‖b‖). It depends only on the angle between the vectors and ranges from −1 to 1. For vectors normalised to length 1 it equals the dot product.

Worked example with made-up three-dimensional vectors: toy = (0.9, 0.1, 0.3), jouet = (0.8, 0.2, 0.4), shoe = (0.1, 0.9, 0.2).

- toy · jouet = 0.72 + 0.02 + 0.12 = 0.86; ‖toy‖ = 0.954, ‖jouet‖ = 0.917; cos = 0.86 / 0.875 = 0.98.
- cos(toy, shoe) = (0.09 + 0.09 + 0.06) / (0.954 × 0.927) = 0.27.

In bag-of-words, "toy" and "jouet" are different columns, so their cosine is 0, the same as "toy" and "shoe".

Embeddings are learned from the **distributional hypothesis**: words that occur in similar contexts have similar meanings (Firth, 1957: "You shall know a word by the company it keeps"). **word2vec** (Mikolov et al., 2013) learns one vector per word by predicting neighbouring words. Its limit: "Maus" gets one vector, whether it is a computer mouse (heading 8471) or a plush toy mouse (9503). Transformer models produce **contextual embeddings**: the vector of a token depends on the whole sentence around it. **Multilingual** models are trained on many languages together, so that "Spielzeug" and "jouet" end up close.

### Why it matters

Embeddings turn meaning into geometry. Once texts are vectors, "find similar decisions", "group decisions by product type" and "use the text as a feature" become standard operations on vectors: nearest neighbours, k-means and linear classifiers (second half of this notebook). Because embedding models are pre-trained on billions of words in many languages, they bring knowledge about synonyms and translations that no labelled dataset of 50,000 decisions contains.

### How it works in Python

In [3]:
import numpy as np


def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))


toy, jouet, shoe = np.array([0.9, 0.1, 0.3]), np.array([0.8, 0.2, 0.4]), np.array([0.1, 0.9, 0.2])
print(round(cosine(toy, jouet), 3), round(cosine(toy, shoe), 3))   # 0.984 0.271
onehot = np.eye(3)                                         # bag-of-words: one column per word
print(cosine(onehot[0], onehot[1]))                        # 0.0 for any two different words


# contextual embeddings: the vector of a token depends on its sentence
def token_vector(sentence, piece):
    vecs = model.encode(sentence, output_value="token_embeddings")      # one vector per token
    tokens = sp_tok.convert_ids_to_tokens(sp_tok(sentence)["input_ids"])
    return vecs[tokens.index(piece)].cpu().numpy()


a = token_vector("passage: wireless mouse for a computer with USB receiver", "▁mouse")
b = token_vector("passage: plush toy shaped like a mouse for children", "▁mouse")
c = token_vector("passage: optical mouse with scroll wheel for laptops", "▁mouse")
print(round(float(cosine(a, b)), 2), round(float(cosine(a, c)), 2))
# 0.92 0.94: the two computer mice are a little closer to each other than to the toy mouse

0.984 0.271
0.0


0.92 0.94


The difference is small for this model: e5 was trained to produce good *sentence* vectors, and its token vectors carry a lot of the whole sentence. The direction is still the expected one.

### In practice

- Grbovic and Cheng (2018) described how Airbnb learns embeddings of listings from users' click sequences and uses them for similar-listing recommendations and search ranking.
- Bolukbasi et al. (2016) showed that word embeddings trained on news text reproduce gender stereotypes ("man is to computer programmer as woman is to homemaker"), a widely cited warning that embeddings inherit the biases of their training text.

> **Caution.** Embeddings capture what texts are *about*. Two descriptions of shoes, one with leather uppers (6403) and one with textile uppers (6404), are very close in embedding space, although the tariff separates them. For fine distinctions that depend on one word, a model trained on labels can beat raw similarity; the last section measures this.

## Attention and pre-training (conceptual)

### Concept

A **transformer** (Vaswani et al., 2017) turns the sequence of token embeddings into a sequence of contextual vectors by stacking layers. The key operation in each layer is **attention**: every token's vector is replaced by a weighted average of the vectors of all tokens in the text, with weights that depend on how relevant each other token is.

In "the boot leaks because it cracked", the vector for "it" should take information from "boot". Attention computes, for each pair of tokens, a relevance score and turns the scores of each row into weights that sum to 1 (softmax). Technically each token has a **query** (what it looks for), a **key** (what it offers) and a **value** (what it passes on); the score of token i for token j is the dot product of query i and key j. A model has several attention "heads" per layer and many layers, so it can track several relations at once.

![Illustrative attention weights for the sentence 'the boot leaks because it cracked'; the row for 'it' puts weight 0.61 on 'boot'](../figures/attention.png)

**Pre-training** teaches the model language before it sees any task. It needs no labels, only text:

- **masked language modelling** (BERT, XLM-RoBERTa): hide 15 % of the tokens and predict them from both sides: "Schuhe mit Oberteil aus [MASK]" → "Leder";
- **next-token prediction** (GPT): predict each token from all tokens before it: "shoes with uppers of" → "leather".

**Fine-tuning** then continues training on a smaller labelled dataset for one task; **contrastive training** on pairs of texts with the same meaning produces sentence-embedding models (below); **instruction tuning** trains a generative model on examples of instructions and good answers so that it follows prompts (Block 2).

### Why it matters

Attention is what bag-of-words lacks: the representation of a word depends on its context, at any distance, so "Leder" can be linked to "Oberteil" and not to "Laufsohle" five words later. Pre-training on unlabelled text is why these models work with few or even no labelled examples, and pre-training on many languages is why one model can compare a Polish and a German description.

### How it works in Python

A toy: three tokens with two-dimensional vectors. Real models use hundreds of dimensions, many heads and learned matrices for queries, keys and values.

In [4]:
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


Q = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])   # queries: what each token looks for
K = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])   # keys: what each token offers
V = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])   # values: what each token passes on
A = softmax(Q @ K.T / np.sqrt(2))                    # attention weights, one row per token
print(A.round(2))
# [[0.46 0.22 0.32]
#  [0.22 0.46 0.32]
#  [0.33 0.33 0.33]]
print((A @ V).round(2))                              # new vector per token = weighted mix of values
# [[2.73 3.73]
#  [3.19 4.19]
#  [3.   4.  ]]

[[0.46 0.22 0.32]
 [0.22 0.46 0.32]
 [0.33 0.33 0.33]]
[[2.73 3.73]
 [3.19 4.19]
 [3.   4.  ]]


> **Note.** Attention weights are tempting to read as explanations ("the model looked at 'boot'"), but they are not a reliable account of why a model made a prediction (Jain and Wallace, 2019). Treat figures like the one above as an illustration of the mechanism.

## Encoder, decoder and encoder–decoder models

### Concept

Transformer models come in three families, depending on which tokens attention may look at and what the model is trained to produce.

| Family | Attention | Pre-training | Typical output | Examples |
|---|---|---|---|---|
| **Encoder** (BERT type) | every token sees all tokens (bidirectional) | masked language modelling | one vector per token or per text | BERT, XLM-RoBERTa, multilingual-e5, German `gbert` |
| **Decoder** (GPT type) | every token sees only earlier tokens (causal) | next-token prediction | generated text, one token at a time | GPT-4o, Llama, Qwen, Mistral |
| **Encoder–decoder** | encoder reads the input; decoder generates and attends to the encoder | reconstruct or transform text | a new text from an input text | T5, BART, Whisper, translation models |

```mermaid
flowchart TB
    subgraph ENC["Encoder (BERT type)"]
        direction LR
        e1["Schuhe aus [MASK]"] --> e2["bidirectional<br/>attention"] --> e3["vectors per token<br/>-> 'Leder'"]
    end
    subgraph DEC["Decoder (GPT type)"]
        direction LR
        d1["shoes with uppers of"] --> d2["causal attention<br/>(only earlier tokens)"] --> d3["next token:<br/>'leather'"]
    end
    subgraph ED["Encoder-decoder (T5 type)"]
        direction LR
        x1["Spielzeugauto aus Kunststoff"] --> x2["encoder"] --> x3["decoder"] --> x4["plastic toy car"]
    end
```

Rules of thumb: use an **encoder** when you need a representation of a text (classification, semantic search, clustering); use a **decoder** when you need to generate text or follow free-form instructions; use an **encoder–decoder** for tasks that map one text to another (translation, summarisation, speech to text). Large decoders can do all of these tasks through prompting, at higher cost.

### Why it matters

Choosing the family decides cost and fit. An encoder such as multilingual-e5-small (118 million parameters, 96 million of them in the token-embedding table for 250,000 tokens) runs on a laptop. A decoder with billions of parameters needs a GPU server or a paid API and returns text that must be parsed. Many practical systems combine both: an encoder for search, a decoder for the answer (Session 15).

### How it works in Python

In [5]:
print(model)          # Transformer (BertModel architecture) -> Pooling (mean) -> Normalize
enc = model[0].auto_model
print(type(enc).__name__, enc.config.num_hidden_layers, enc.config.hidden_size)
# BertModel 12 384: an encoder with 12 layers and 384-dimensional vectors
n_all = sum(p.numel() for p in enc.parameters())
n_emb = enc.embeddings.word_embeddings.weight.numel()
print(round(n_all / 1e6, 1), round(n_emb / 1e6, 1))
# 117.7 96.0: million parameters in total and in the token-embedding table alone

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
BertModel 12 384
117.7 96.0


> **Tip.** In the Hugging Face model hub, the model card states the architecture. "fill-mask" or "feature-extraction" usually means encoder; "text-generation" means decoder; "text2text-generation", "translation" or "summarization" usually means encoder–decoder.

> 📖 **Read more:** [extras/03-hf-course-transformer-pipelines.ipynb](extras/03-hf-course-transformer-pipelines.ipynb) (Hugging Face course): runs one encoder, decoder and encoder–decoder task each with `pipeline()`; it downloads several models, so pick the small ones.

## Embedding models (sentence-transformers)

### Concept

A **sentence-embedding model** maps a whole text to one vector so that texts with similar meaning get a high cosine similarity. **Sentence-BERT** (Reimers and Gurevych, 2019) takes a pre-trained encoder, averages its token vectors (**mean pooling**) and fine-tunes it on millions of pairs of texts that mean the same (question and answer, duplicate questions, translations), so that paired texts move close together and unrelated texts move apart (**contrastive training**).

The `sentence-transformers` library provides hundreds of such models. Two small multilingual ones fit the case study:

| Model | Size | Dimensions | Input limit | Notes |
|---|---|---|---|---|
| `intfloat/multilingual-e5-small` (Wang et al., 2024) | about 470 MB | 384 | 512 tokens | inputs need a prefix: `"query: "` for questions, `"passage: "` for documents |
| `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` | about 470 MB | 384 | 128 tokens | no prefix; truncates 65 % of the descriptions |

The course uses multilingual-e5-small because its 512-token limit covers 99 % of the descriptions. Larger models are compared on the **MTEB** benchmark (Muennighoff et al., 2023). The option `normalize_embeddings=True` scales every vector to length 1, so the matrix product `E @ E.T` gives all pairwise cosine similarities.

### Why it matters

Sentence embeddings compare meaning rather than shared words, across languages. "Spielzeugauto aus Kunststoff", "Voiture jouet en matière plastique" and "Toy car of plastics" share no word (TF-IDF cosine 0), but their embeddings are close. Encoding is a one-time cost: the vectors can be stored and reused.

### How it works in Python

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sents = ["passage: Spielzeugauto aus Kunststoff, mit Rädern",
         "passage: Voiture jouet en matière plastique, avec roues",
         "passage: Toy car of plastics with wheels",
         "passage: Damenschuhe mit Oberteil aus Leder",
         "passage: Ladies' shoes with leather uppers"]
E = model.encode(sents, normalize_embeddings=True)
print(E.shape)                                   # (5, 384): one vector per text
print((E @ E.T).round(2))                        # cosine similarities (vectors have length 1)
# [[1.   0.94 0.91 0.86 0.81]
#  [0.94 1.   0.91 0.83 0.82]
#  [0.91 0.91 1.   0.77 0.8 ]
#  [0.86 0.83 0.77 1.   0.9 ]
#  [0.81 0.82 0.8  0.9  1.  ]]
print(cosine_similarity(TfidfVectorizer().fit_transform([s[9:] for s in sents])).round(2)[0])
# [1.  0.  0.  0.3 0. ]: with TF-IDF, the German toy car only resembles the German shoes ('aus', 'mit')

(5, 384)
[[1.   0.94 0.91 0.86 0.81]
 [0.94 1.   0.91 0.83 0.82]
 [0.91 0.91 1.   0.77 0.8 ]
 [0.86 0.83 0.77 1.   0.9 ]
 [0.81 0.82 0.8  0.9  1.  ]]
[1.  0.  0.  0.3 0. ]


The three toy cars (0.91–0.94) are closer to each other than to the shoes, and the German and English shoes are close (0.90). Note the narrow range: for e5, unrelated texts still score about 0.75–0.8. Compare similarities with each other, never with a fixed threshold such as 0.5.

> **Warning.** Text longer than the model's limit (512 tokens for multilingual-e5-small, 128 for paraphrase-multilingual-MiniLM) is silently cut off. Check `model.max_seq_length` before you rely on the vector of a long description; for long documents, embed passages (Session 15).

> **Tip.** Forgetting the e5 prefixes does not raise an error, but it lowers retrieval quality. Use `"passage: "` for the decisions you store and `"query: "` for the text you search with; for classification features, the e5 authors recommend `"query: "` for all texts.

## What embeddings are good for

The rest of the notebook uses the vectors for three tasks and compares each with TF-IDF on the same decisions: 10,000 training decisions from 2017–2021 and 2,000 validation decisions from 2022–2023 (validation by time, as in Session 13).

```mermaid
flowchart LR
    R["Decisions<br/>(22 languages)"] --> M["Embedding model<br/>(encode once)"]
    M --> E[("Vectors<br/>12,000 x 384<br/>store as .npy")]
    E --> S["Semantic search<br/>cosine + top-k"]
    E --> C["Clustering<br/>k-means, UMAP plot"]
    E --> F["Features<br/>linear classifier"]
    Q["Query in any language"] --> M2["same model"] --> S
```

In [7]:
import time

year = decisions["start_date"].dt.year
train = decisions[year <= 2021].sample(10_000, random_state=0)     # validation by time, as before
valid = decisions[year >= 2022].sample(2_000, random_state=0)

t0 = time.perf_counter()
E_tr = model.encode(("passage: " + train["description"]).tolist(), batch_size=64, normalize_embeddings=True)
E_va = model.encode(("query: " + valid["description"]).tolist(), batch_size=64, normalize_embeddings=True)
print(E_tr.shape, E_va.shape)          # (10000, 384) (2000, 384)
print(f"{time.perf_counter() - t0:.0f} s")   # about a minute on a laptop

(10000, 384) (2000, 384)
36 s


## Semantic search

### Concept

**Semantic search** retrieves documents by meaning instead of shared keywords, in two phases:

1. **Indexing** (once): embed every document and store the vectors.
2. **Querying** (per request): embed the query with the same model, compute the cosine similarity between the query vector and every document vector, and return the k documents with the highest similarity (**top-k**).

With vectors of length 1, step 2 is one matrix–vector product. For millions of documents, **approximate nearest-neighbour indexes** (for example FAISS) find the top-k without comparing the query with every vector.

Worked example: three decision vectors (already normalised) r1 = (0.8, 0.6), r2 = (0.6, 0.8), r3 = (−0.6, 0.8) and a query q = (1, 0). The similarities are q·r1 = 0.8, q·r2 = 0.6, q·r3 = −0.6, so the ranking is r1, r2, r3.

**Keyword search** with TF-IDF works the same way with sparse vectors; it finds only documents that share words with the query, which across languages means almost none. A useful special case: use the description of a **new** decision as the query. The headings of its nearest neighbours are candidate headings (a **k-nearest-neighbour classifier**), and the share of queries whose true heading appears among the k neighbours ("hit@k") measures how useful the search is for a customs officer.

### Why it matters

A customs officer in Germany who classifies a new product wants to see how other member states classified similar goods, whatever the language of their decisions. Semantic search makes this possible without translating anything, and it is the retrieval step of retrieval-augmented generation (Session 15). Keyword search remains useful for exact terms (product codes, brand names, CAS numbers of chemicals), and many systems combine both (**hybrid search**).

### How it works in Python

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

german = (train["language"] == "de").to_numpy()            # search only the German decisions
tfidf = TfidfVectorizer(min_df=2, sublinear_tf=True).fit(train["description"])
T_de = tfidf.transform(train["description"][german])
heads_de = train["heading"].to_numpy()[german]


def search(query, k=5):
    q = model.encode("query: " + query, normalize_embeddings=True)
    emb_scores = E_tr[german] @ q                          # cosine similarity with every decision
    kw_scores = (T_de @ tfidf.transform([query]).T).toarray().ravel()
    for name, scores in [("embedding", emb_scores), ("tf-idf", kw_scores)]:
        top = np.argsort(-scores)[:k]
        print(name, [(heads_de[i], round(float(scores[i]), 2)) for i in top])


search("disposable face masks")                            # an English query, German documents
# embedding [('6307', 0.81), ('3401', 0.8), ('3401', 0.8), ('3214', 0.8), ('9033', 0.79)]
# tf-idf    [('9018', 0.0), ('9403', 0.0), ...]            no shared word: all scores are 0
search("toys for children")
# embedding [('9503', 0.83), ('9503', 0.83), ('9503', 0.83), ('9503', 0.83), ('9503', 0.83)]
# tf-idf    [('8471', 0.04), ('8536', 0.03), ...]

# nearest neighbours as candidate headings for the 2,000 validation decisions
S = E_va @ E_tr.T
nn = np.argsort(-S, axis=1)[:, :5]
heads = train["heading"].to_numpy()
print(round(float(np.mean(heads[nn[:, 0]] == valid["heading"].to_numpy())), 3),       # 1-NN
      round(float(np.mean([h in heads[r] for h, r in zip(valid["heading"], nn)])), 3))  # hit@5
# 0.56 0.718
same_lang = train["language"].to_numpy()[nn] == valid["language"].to_numpy()[:, None]
print(round(float(same_lang.mean()), 3))     # 0.971: neighbours are almost always in the same language

# cross-lingual: non-German queries searched only among German decisions
other = (valid["language"] != "de").to_numpy()
nn_x = np.argsort(-(E_va[other] @ E_tr[german].T), axis=1)[:, :5]
print(round(float(np.mean([h in heads_de[r] for h, r in zip(valid["heading"][other], nn_x)])), 3))
# 0.416 hit@5 with embeddings
T_tr, T_va = tfidf.transform(train["description"]), tfidf.transform(valid["description"])
nn_kx = np.argsort(-(T_va[other] @ T_tr[german].T).toarray(), axis=1)[:, :5]
print(round(float(np.mean([h in heads_de[r] for h, r in zip(valid["heading"][other], nn_kx)])), 3))
# 0.166 hit@5 with TF-IDF (shared numbers, brand names and loan words)

embedding [('6307', 0.81), ('3401', 0.8), ('3401', 0.8), ('3214', 0.8), ('9033', 0.79)]
tf-idf [('9018', 0.0), ('9403', 0.0), ('0604', 0.0), ('9031', 0.0), ('8428', 0.0)]
embedding [('9503', 0.83), ('9503', 0.83), ('9503', 0.83), ('9503', 0.83), ('9503', 0.83)]
tf-idf [('8471', 0.04), ('8536', 0.03), ('0604', 0.0), ('9031', 0.0), ('8428', 0.0)]


0.56 0.718
0.971


0.416


0.166


Three findings. First, the English queries find German mask and toy decisions; TF-IDF finds nothing. Second, when all languages are allowed, 97 % of the neighbours share the query's language: the model places texts of one language closer together than translations, so a cross-lingual search must be asked for explicitly, for example by filtering the corpus. Third, restricted to German decisions, embeddings find the right heading among five neighbours for 42 % of the non-German queries, two and a half times as often as TF-IDF.

- Thakur et al. (2021) compared retrieval methods on the 18 datasets of the BEIR benchmark and found that the keyword method BM25 remained a strong baseline across domains, while dense embedding retrieval was better in some domains and worse in others. This is the main argument for hybrid search (Session 15).

> **Warning.** Query and documents must be embedded with the **same model** (and the same version and prefixes). Vectors from different models live in different spaces; comparing them gives meaningless numbers without an error message.

> 📖 **Read more:** [extras/04-sbert-retrieve-rerank-wikipedia.ipynb](extras/04-sbert-retrieve-rerank-wikipedia.ipynb) (sentence-transformers): semantic search over Simple English Wikipedia with a bi-encoder, re-ranking of the hits with a cross-encoder, and BM25 keyword results side by side for each query.

## Clustering decisions

### Concept

**Clustering** groups documents without labels (Session 11). The standard recipe on embeddings:

1. Embed and normalise the documents (for normalised vectors, Euclidean distance and cosine similarity rank pairs identically).
2. Run k-means for several values of k; inspect the silhouette score and, more importantly, the content of the clusters.
3. **Describe** each cluster by its most distinctive words (TF-IDF mean in the cluster divided by the overall mean), its most frequent heading and language.
4. **Visualise** with UMAP or PCA in two dimensions (Session 11), keeping in mind that the 2-D picture distorts distances.

![UMAP projection of 1,500 decisions from five chapters: German decisions form one island per chapter, while French, English and other languages share a central region that is still ordered by chapter](../figures/embedding_space.png)

Colour by chapter (left): footwear, toys, knitted clothing, beverages and electrical machinery form separate groups. Colour by language (right): every German chapter forms its own island, while the decisions in other languages sit together in a central region, still ordered by chapter. Both **topic and language** shape the space.

### Why it matters

Clustering embeddings is a fast way to discover what a large text collection is about and can reveal problems in the labels: a decision far from the other decisions of its heading is worth a second look, the idea of the distance-based anomaly detection of Session 11. Because language also shapes the space, clusters must be checked for whether they reflect goods or only languages.

### How it works in Python

In [9]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

shoes = (train["chapter"] == "64").to_numpy()             # the footwear chapter: 368 decisions
km = KMeans(n_clusters=6, n_init=10, random_state=0).fit(E_tr[shoes])
print(shoes.sum(), round(silhouette_score(E_tr[shoes], km.labels_), 3),
      round(adjusted_rand_score(train["heading"][shoes], km.labels_), 2))
# 368 0.115 0.36: weak cluster structure; partial agreement with the six headings
words = TfidfVectorizer(min_df=3).fit(train["description"][shoes])
W = words.transform(train["description"][shoes])
names = words.get_feature_names_out()
overall = np.asarray(W.mean(axis=0)).ravel()
for c in range(6):
    in_c = km.labels_ == c
    lift = np.asarray(W[in_c].mean(axis=0)).ravel() / (overall + 1e-9)   # distinctive words
    lift[np.asarray((W[in_c] > 0).sum(axis=0)).ravel() < 5] = 0          # ignore very rare words
    part = train[shoes][in_c]
    print(c, in_c.sum(), list(names[np.argsort(-lift)[:4]]),
          part["heading"].value_counts().index[0], part["language"].value_counts().index[0])
# 0 98 ['wade', 'jedoch', '9110', '9105'] 6403 de
# 1 68 ['männerschuhe', '9998', '9950', '9991'] 6403 de
# 2 17 ['obuv', 'údajů', 'jedná', 'dle'] 6403 cs          <- a Czech cluster
# 3 121 ['hausschuhe', 'ausstattung', 'labilem', 'weichem'] 6404 de   <- slippers, textile uppers
# 4 17 ['with', 'and', 'the', 'sole'] 6403 en             <- an English cluster
# 5 47 ['et', 'matière', 'une', 'semelle'] 6405 fr        <- a French cluster

368 0.115 0.36
0 98 ['wade', 'jedoch', '9110', '9105'] 6403 de
1 68 ['männerschuhe', '9998', '9950', '9991'] 6403 de
2 17 ['obuv', 'údajů', 'jedná', 'dle'] 6403 cs
3 121 ['hausschuhe', 'ausstattung', 'labilem', 'weichem'] 6404 de
4 17 ['with', 'and', 'the', 'sole'] 6403 en
5 47 ['et', 'matière', 'une', 'semelle'] 6405 fr


Cluster 3 collects slippers ("Hausschuhe"), almost all heading 6404 (103 of 121); clusters 2, 4 and 5 are defined by language. An adjusted Rand index of 0.36 means partial agreement with the headings. Choose k by whether the clusters are useful and distinct when you read them, not by the silhouette score alone.

- BERTopic (Grootendorst, 2022) builds topic models this way (embeddings, UMAP, density-based clustering, then class-based TF-IDF to describe clusters).

> **Caution.** Clusters are not facts about the data; they depend on the model, k, the random seed and the preprocessing. With multilingual data, cross-tabulate clusters against `language` before you interpret them as product groups.

> 📖 **Read more:** [extras/05-openai-cookbook-clustering.ipynb](extras/05-openai-cookbook-clustering.ipynb) (OpenAI Cookbook): k-means on precomputed embeddings of food reviews with a t-SNE plot, and naming each cluster by letting a GPT model summarise sample reviews (that last part needs an OpenAI key). The scikit-learn example in Session 13 ([extras notebook 04](../../13-classical-nlp/workbooks/extras/04-sklearn-document-clustering-lsa.ipynb)) clusters with LSA vectors instead of neural embeddings.

## Embedding features versus TF-IDF

### Concept

Embeddings can replace the TF-IDF matrix as input to any classifier. Each description becomes a dense vector of 384 numbers, and the linear classifier learns one weight per dimension and heading. The embedding model is used **frozen**: only the classifier on top is trained. (Fine-tuning the whole model on the labels belongs to module 3.4.) A fair comparison uses the same decisions, split, classifier type and metrics:

| Representation | Dimensions | Knows translations? | Sees one decisive word? | Cost |
|---|---|---|---|---|
| TF-IDF word unigrams | about 30,000 (sparse) | no | yes (one column per word) | very low |
| multilingual-e5 embeddings | 384 (dense) | yes | weakly (averaged into 384 numbers) | encoding time |
| both, concatenated | about 30,000 + 384 | yes | yes | both |

### Why it matters

Pre-trained embeddings are expected to help most when labelled data are scarce. Tariff classification is a hard test for this expectation: the classes are fine-grained (leather versus textile uppers), and the decisive information is often one word or one number that an averaged vector blurs.

### How it works in Python

In [10]:
from scipy.sparse import csr_matrix, hstack
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score


def svm():
    return SGDClassifier(loss="hinge", alpha=1e-5, max_iter=20, tol=None, random_state=0, n_jobs=-1)


def report(name, clf, X):
    p = clf.predict(X)
    print(name, round(accuracy_score(valid["heading"], p), 3),
          round(f1_score(valid["heading"], p, average="macro"), 3))


report("embeddings", svm().fit(E_tr, train["heading"]), E_va)    # embeddings 0.567 0.303
report("tf-idf    ", svm().fit(T_tr, train["heading"]), T_va)    # tf-idf     0.662 0.411
H_tr = hstack([T_tr, csr_matrix(E_tr)]).tocsr()                  # both side by side
H_va = hstack([T_va, csr_matrix(E_va)]).tocsr()
report("both      ", svm().fit(H_tr, train["heading"]), H_va)    # both       0.675 0.423

embeddings 0.567 0.303


tf-idf     0.662 0.411


both       0.675 0.423


| Training decisions | 1-NN on embeddings | 1-NN on TF-IDF |
|---|---|---|
| 500 | 0.27 | 0.34 |
| 2,000 | 0.40 | 0.47 |
| 10,000 | 0.56 | 0.62 |

All values are accuracy on the same 2,000 validation decisions; the learning curve comes from [case study 04](04-case-study-embeddings-vs-tfidf.ipynb). TF-IDF is better at every training size: the expected advantage of embeddings with few labels does not appear, because with 1,000 headings and 500 examples most headings have no example at all. The accuracies are lower than in Session 13 (0.77) because only 10,000 training decisions are used. With 2,000 validation decisions, differences below about two points are within noise; a bootstrap interval (Session 7) makes this precise.

- Tunstall et al. (2022) showed with SetFit that sentence-transformers fine-tuned on as few as eight labelled examples per class can compete with much larger models on tasks with a handful of classes: fine-tuning, not frozen vectors, is what makes embeddings strong classifiers.

> **Important.** Report the TF-IDF baseline next to every embedding result. On the EBTI data, a free, fast TF-IDF model beats frozen multilingual embeddings as classification features. The embeddings earn their cost through cross-lingual search, exploration, and as candidate retrieval for a language model (Block 2, Session 15).

> **Tip.** Encoding is the expensive step. Encode once and save the vectors (`np.save`), together with the model name, the prefix and the decision ids, instead of re-encoding in every run.

## Check your understanding

1. Why can a subword tokeniser represent any word, while `CountVectorizer` cannot? What is the price?
2. Compute the cosine similarity of (1, 0, 1) and (1, 1, 0) by hand.
3. Explain in two sentences what attention does, using the example "Schuhe mit Oberteil aus Leder und Laufsohle aus Kunststoff".
4. Which model family would you choose for (a) finding decisions similar to a new request, (b) writing a short justification for a proposed heading, (c) translating a Polish description into English? Why?
5. 97 % of the nearest neighbours share the query's language. What does this mean for a customs officer who wants to see decisions of other member states?
6. Embeddings lose to TF-IDF as classification features even with 500 training decisions. Give two reasons specific to tariff classification.

## Further reading

- Alammar, J. (2018). *The Illustrated Transformer*. https://jalammar.github.io/illustrated-transformer/
- Hugging Face. *LLM Course*, chapters 1 (transformer models) and 6 (tokenisers). https://huggingface.co/learn/llm-course
- Reimers, N. *Sentence-Transformers documentation*: semantic search, clustering and multilingual models. https://sbert.net/
- Wang, L. et al. (2024). Multilingual E5 text embeddings: a technical report. arXiv:2402.05672. https://arxiv.org/abs/2402.05672
- Muennighoff, N., Tazi, N., Magne, L. and Reimers, N. (2023). MTEB: massive text embedding benchmark. *Proceedings of EACL 2023*. Leaderboard: https://huggingface.co/spaces/mteb/leaderboard
- Thakur, N. et al. (2021). BEIR: a heterogeneous benchmark for zero-shot evaluation of information retrieval models. *NeurIPS 2021 Datasets and Benchmarks*. https://arxiv.org/abs/2104.08663